In [ ]:
"""! @brief Optimization with Gradient Descent, RMSProp and CMA-ES"""
# @file TP1.ipynb
#
# @mainpage Optimization Project
#
# @section description_main Description
# This project evaluates and compares three optimization algorithms:
# Gradient Descent, RMSProp, and CMA-ES. The goal is to optimize three
# different objective functions while using Optuna to tune hyperparameters.
#
# @section libraries_main Libraries
# - torch (https://docs.pytorch.org/docs/stable/index.html)
#   - Access to tensor and all its operations
# - numpy (https://numpy.org/doc/)
#   - Numerical computing library for array operations
# - matplotlib.pyplot (https://matplotlib.org/stable/index.html)
#   - Access to plot, contour and other visual options
# - optuna (https://optuna.readthedocs.io/en/stable/index.html)
#   - Used to optimize hyperparameters automatically
# - pandas (https://pandas.pydata.org/docs/)
#   - Used to generate tables
# - math (https://docs.python.org/3/library/math.html)
#   - Access to inf
#
# @section authors_main Authors
# - Alejandro Cerdas
# - Kener Castillo
# - Pablo Perez

# Instalación de Optuna
!pip install optuna

# Imports

In [ ]:
# @brief Imports and global constants declarations

# Imports
import torch
import numpy as np
import matplotlib.pyplot as plt
import optuna
import pandas as pd
import math


# Global Constants
## Number pi of torch
PI = torch.pi
## Lower Bound used in the plots
MIN_PLOT = -1e1
## Upper Bound used in the plots
MAX_PLOT = 1e1
## Number of points equal spaced for the plot
NUMBER_POINTS= 600
## Number of images of the function to show
LEVELS = 20
## Random points used to initialize the searches
INITIAL_POINTS = [torch.randint(-10, 10, (2,)) for _ in range(10)]

# Utils

In [ ]:
# @brief Definition of auxiliar classes

class Function:
    """! The Function base class.
    Defines the function to optimize and its gradient.
    """
    def __init__(self, f, gradient, n):
        """! The function base class initializer.

        @param f The function to be optimized.
        @param gradient The gradient of the function.
        @param n ID of the function (0, 1, 2).

        @return an instance of the Function class
        """
        # Stores the objective function
        self.ob_f = f

        # Stores the gradient function
        self.ob_gradient = gradient

        # Stores the function identifier
        self.n = n

    def f(self, x, y, lib):
        """! Evaluation of the function.
        @param x The x coordinate.
        @param y The y coordinate.
        @param lib The library to use (Torch or Numpy).

        @return Value of the function.
        """
        return self.ob_f(x, y, lib)

    def gradient(self, vec: torch.Tensor):
        """! Evaluation of the gradient of the function.
        @param vec Vector to evaluate

        @return The gradient as a vector
        """
        return self.ob_gradient(vec[0], vec[1])

class Plot:
    """! Class responsible for generating different graphical representations of a two-variable function.

    This class builds a grid over the XY plane using global constants and
    provides methods to generate 3D surfaces, contour plots, contour plots
    with points, surfaces with points, and learning curves.
    """

    def __init__(self):
        """! Initializes the grid used for plotting.

        @return An instance of Plot with points defined in a range and a 2D meshgrid.

        @note This method depends on the following global constants:
              - MIN_PLOT
              - MAX_PLOT
              - NUMBER_POINTS

        """
        # Generate NUMBER_POINTS points in the range of MIN_PLOT and MAX_PLOT
        self.x = np.linspace(MIN_PLOT, MAX_PLOT, NUMBER_POINTS)

        # Generate NUMBER_POINTS points in the range of MIN_PLOT and MAX_PLOT
        self.y = np.linspace(MIN_PLOT, MAX_PLOT, NUMBER_POINTS)

        # Creates 2 numpy 2D arrays with all possible pair of the elements in x and y
        self.X, self.Y = np.meshgrid(self.x, self.y)

    def make_surface(self, fun: Function, f_name):
        """! Generates a 3D surface plot of the function.

        @param fun Function object that provides method `f(x, y, np)`.
        @param f_name Name of the function, used in the plot title.
        """
        # Evaluates the function over the grid
        Z = fun.f(self.X, self.Y, np)

        # Creates a new figure for the 3D plot
        fig1 = plt.figure()

        # Adds a 3D subplot to the figure
        ax1 = fig1.add_subplot(111, projection='3d')

        # Plots the surface using the computed Z values
        ax1.plot_surface(self.X, self.Y, Z, cmap='viridis')

        # Labels the axes
        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        ax1.set_zlabel('f(x,y)')

        # Sets the plot title
        plt.title("Surface of function " + f_name)

    def make_contour(self, fun: Function, f_name):
        """! Generates a 2D contour plot of the function.

        @param fun Function object that provides method `f(x, y, np)`.
        @param f_name Name of the function, used in the plot title.

        @note The number of contour levels depends on the global constant `LEVELS`.
        """
        # Evaluates the function over the grid
        Z = fun.f(self.X, self.Y, np)

        # Creates a new figure for the contour plot
        fig2 = plt.figure()

        # Draws contour lines using the specified number of levels
        plt.contour(self.X, self.Y, Z, levels=LEVELS, cmap='plasma')

        # Adds a color scale to interpret function values
        plt.colorbar()

        # Sets plot title and axis labels
        plt.title("Contour of function " + f_name)
        plt.xlabel('x')
        plt.ylabel('y')

    def make_contour_p(self, fun: Function, x, y, name, f_name):
        """! Generates a contour plot with points overlaid.

        @param fun Function object that provides method `f(x, y, np)`.
        @param x Array or list of x-coordinates of the points.
        @param y Array or list of y-coordinates of the points.
        @param name Name of the method or algorithm.
        @param f_name Name of the function, used in the plot title.
        """
        # Evaluates the function over the grid
        Z = fun.f(self.X, self.Y, np)

        # Creates a new figure for the contour plot
        fig3 = plt.figure()

        # Draws contour lines of the function
        plt.contour(self.X, self.Y, Z, levels=25, cmap='plasma')

        # Adds a colorbar to represent function values
        plt.colorbar()

        # Plots the optimization path points with color indicating iteration order
        plt.scatter(x, y, c=[i for i in range(len(x))], cmap='inferno', s=50, label='Points')

        # Sets plot title and axis labels
        plt.title("Contour with points of " + name + " on function " + f_name)
        plt.xlabel('x')
        plt.ylabel('y')

        # Displays the legend
        plt.legend()

    def make_surface_p(self, fun: Function, x, y, z, name, f_name):
        """! Generates a 3D surface plot with points overlaid.

        Draws the function surface and overlays 3D points, useful for
        visualizing optimization steps or sampled solutions.

        @param fun Function object that provides method `f(x, y, np)`.
        @param x Array or list of x-coordinates.
        @param y Array or list of y-coordinates.
        @param z Array or list of z-values (f(x, y)).
        @param name Name of the method or algorithm.
        @param f_name Name of the function, used in the plot title.
        """
        # Evaluates the function over the grid
        Z = fun.f(self.X, self.Y, np)

        # Creates a new figure for the 3D plot
        fig4 = plt.figure()

        # Adds a 3D subplot to the figure
        ax1 = fig4.add_subplot(111, projection='3d')

        # Plots the function surface with transparency
        ax1.plot_surface(self.X, self.Y, Z, cmap='viridis', alpha=0.5)

        # Plots optimization points colored by their function value
        ax1.scatter(x, y, z, c=z, cmap='coolwarm', s=60)

        # Labels the axes
        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        ax1.set_zlabel('f(x,y)')

        # Sets the plot title
        plt.title("Surface with points of " + name + " on function " + f_name)

    @staticmethod
    def make_plot(x, y, name, f_name):
        """! Generates a learning curve plot.

        Plots the evolution of the function value across iterations
        of an algorithm.

        @param x Array or list of iteration indices.
        @param y Array or list of function values at each iteration.
        @param name Name of the method or algorithm.
        @param f_name Name of the function, used in the plot title.
        """
        # Creates a new figure for the plot
        fig5 = plt.figure()

        # Plots function values across iterations
        plt.plot(x, y, marker='o')

        # Sets plot title and axis labels
        plt.title("Learning curve of " + name + " on function " + f_name)
        plt.ylabel('f(x,y)')
        plt.xlabel('Iteration')

    @staticmethod
    def show():
        """! Displays all generated plots.
        """

        # Display the plot
        plt.show()

# Object Plot initialization
plot = Plot()

class Table:
    """! The Table class
    Defines the functions used to generate, update and show a pandas dataframe
    """
    def __init__(self, data: pd.DataFrame, names):
        """! The Table class initializer
        @param data The panda data frame
        @param names The names used to define the columns in the table

        @return An instance of the Table class initialized with a dataframe with the given columns names
        """

        # The dataframe of the table
        self.data = data

        # Initialize all the columns with the given names
        for name in names:
            self.data[name] = []

    def add(self, run, name, value):
        """! Add a value in a specific column of the table
        @param run Row in the table
        @param name Column in the table
        @param value Value to add in the table
        """

        # Rounds float values for consistency
        if isinstance(value, float):
            value = round(value, 6)

        # Inserts or updates the value in the table
        self.data.loc[run, name] = value

    def show(self):
        """! Show the table
        """

        # Show the dataframe
        print(self.data)

# Funcion 0
No convexa, pero por rangos si, además no tiene un mínimo absoluto, solo puntos silla periódicos.

In [ ]:
"""! @brief Declaration of the function number 0 and its gradient. """

def f0(x, y, lib):
    """! The function 0.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.
    @param lib The library to use sin.

    @return Value of the function in x, y.
    """

    # Evaluate de function in x,y
    return lib.sin(x + y) + (x - y) ** 2 - 1.5 * x + 2.5 * y + 1

def gradient0(x, y):
    """! The gradient of the f0 function.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.

    @return A tensor as the gradient in x, y
    """

    # Partial derivative with respect to x
    nx = torch.cos(x + y) + 2*(x - y) - 1.5

    # Partial derivative with respect to y
    ny = torch.cos(x + y) - 2*(x - y) + 2.5

    # Returns gradient vector
    return torch.tensor([nx, ny])


# Initializes function f0 with its gradient
func0 = Function(f0, gradient0, "0")

# Plots the 3D surface of the function
plot.make_surface(func0, func0.n)

# Plots the contour of the function
plot.make_contour(func0, func0.n)

# Displays all generated plots
plot.show()

# Funcion 1
No es convexa, si tiene punto mínimo absoluto, en (1,1) y tiene múltiples puntos silla.

In [ ]:
"""! @brief Declaration of the function number 1 and its gradient. """

def gradient1(x,y):
    """! The gradient of the f1 function.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.

    @return A tensor as the gradient in x, y
    """

    # Partial derivative with respect to x
    nx = (6*PI*torch.sin(3*PI*x)*torch.cos(3*PI*x))+(2*(x-1)*(1+(torch.sin(3*PI*y)**2)))

    # Partial derivative with respect to y
    ny = ((x-1)**2)*(6*PI*torch.sin(3*PI*y)*torch.cos(3*PI*y))+(2*(y-1)*(1+(torch.sin(2*PI*y)**2)))+((y-1)**2)*(4*PI*torch.sin(2*PI*y)*torch.cos(2*PI*y))

    # Returns gradient vector
    return torch.tensor([nx, ny])

def f1(x, y, lib):
    """! The function 1.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.
    @param lib The library to use sin.

    @return Value of the function in x, y.
    """

    # Evaluate de function in x,y
    return (lib.sin(3 * lib.pi * x) ** 2) + ((x - 1) ** 2) * (1 + (lib.sin(3 * lib.pi * y) ** 2)) + ((y - 1) ** 2) * (1 + lib.sin(2 * lib.pi * y) ** 2)

# Initializes function f1 with its gradient
func1 = Function(f1,gradient1,"1")

# Plots the 3D surface of the function
plot.make_surface(func1, func1.n)

# Plots the contour of the function
plot.make_contour(func1, func1.n)

# Displays all generated plots
plot.show()

# Funcion 2
No es convexa, tiene 4 puntos mímimos absolutos, tiene un mínimo local

In [ ]:
"""! @brief Declaration of the function number 2 and its gradient. """

def gradient2(x,y):
    """! The gradient of the f2 function.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.

    @return A tensor as the gradient in x, y
    """

    # Partial derivative with respect to x
    nx = 2*(x**2 + y - 11)*(2*x) + 2*(x + y**2 - 7)

    # Partial derivative with respect to y
    ny = 2*(x**2 + y - 11) + 2*(x + y**2 - 7)*(2*y)

    # Returns gradient vector
    return torch.tensor([nx, ny])

def f2(x,y,lib):
    """! The function 2.
    @param x A number as the X coordinate.
    @param y A number as the Y coordinate.
    @param lib The library to use sin.

    @return Value of the function in x, y.
    """
    return (x ** 2 + y - 11) ** 2 + ((x + y ** 2 - 7) ** 2)

# Initializes function f2 with its gradient
func2 = Function(f2,gradient2,"2")

# Plots the 3D surface of the function
plot.make_surface(func2,func2.n)

# Plots the contour of the function
plot.make_contour(func2,func2.n)

# Displays all generated plots
plot.show()

# Descenso Gradiente

In [ ]:
"""! @brief Declaration of the class GradientDescent"""

class GradientDescent:
    """! The Gradient Descent class.
    The Gradient Descent algorithm, the calibration of its parameters with Optuna and analysis of the best run.
    """
    def __init__(self, fun: Function, table : Table):
        """! Initializes the gradient descent
        @param fun The function to optimize.
        @param table A panda table to show data of the runs of the algorithm.

        @return Initialization of the class.
        """
        # Number of iterations
        self.T = 25

        # Function to be optimized
        self.fun = fun

        # Initial point of the algorithm
        self.x0 = torch.tensor([2, 1])

        # Table to store results
        self.table = table

    def descent(self, alpha: float, x_t):
        """! Executes the algorithm.
        @param alpha Parameter alpha to use.
        @param x_t Tensor as an arbitrary initial point.

        @return An array of the visited points and its function value.
        """
        pts=[]
        img=[]
        for i in range(self.T):
            x_t = x_t - alpha * self.fun.gradient(x_t)
            #x_t = torch.clamp(x_t, -10, 10)
            pts.append(x_t.tolist())
            val=self.fun.f(x_t[0], x_t[1], torch)
            img.append(val.item())
            if torch.isnan(val) or torch.isinf(val):
                pts.append([float("inf"), float("inf")])
                img.append(float("inf"))
                break
        return pts, img

    def objetive(self, trial):
        """! Function for Optuna to optimize.
        @param trial An Optuna class.

        @return value of the function in this trial.
        """

        # Alpha suggested by Optuna in the range [1e-5, 1e-1]
        alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True)

        # Visited values of the function.
        _, z= self.descent(alpha, self.x0)

        # Return the last value
        return z[-1]

    def study(self):
        """ ! function to calibrate parameters with Optuna.
        @return The best alpha found as a dictionary.
        """

        # Arbitrary initial point for the algorithm.
        self.x0=torch.tensor([5.0, 5.0])

        # To minimize the function.
        stud = optuna.create_study(direction="minimize")

        # Objective function and number of trials.
        stud.optimize(self.objetive, n_trials=50)

        # Print the best parameters found.
        print(stud.best_params)

        # Visualizations of the process provided by Optuna.
        optuna.visualization.plot_optimization_history(stud).show()
        optuna.visualization.plot_slice(stud, params=["alpha"]).show()

        # Return best parameters found
        return stud.best_params

    def show_plots(self, alpha, init):
        """! Generate plots for the best initial points

        @param alpha The hyperparameter alpha of the algorithm.
        @param init The initial point of the algorithm.
        """
        # Executes the optimization algorithm
        pts, img = self.descent(alpha, init)

        # Plots contour with optimization trajectory
        plot.make_contour_p(
            self.fun,
            [x[0] for x in pts],
            [x[1] for x in pts],
            "Descenso de Gradiente",
            self.fun.n
        )

        # Plots surface with optimization points
        plot.make_surface_p(
            self.fun,
            [x[0] for x in pts],
            [x[1] for x in pts],
            img,
            "Descenso de Gradiente",
            self.fun.n
        )

        # Plots learning curve (function value vs iterations)
        plot.make_plot(
            [i for i in range(len(pts))],
            [i for i in img],
            "Descenso de Gradiente",
            self.fun.n
        )

        # Displays all generated plots
        plot.show()

    def run(self):
        """! Run 10 times the algorithm with the optimized parameters and show the best one with grophs.
        """
        best=self.study()
        alpha=best["alpha"]
        it=0
        best_dist = [math.inf, math.inf]
        best_init = torch.tensor([math.inf, math.inf])
        for i in INITIAL_POINTS:
            pts, img=self.descent(alpha, i)
            idx=img.index(min(img))
            self.table.add(it, "Iteration", idx)
            self.table.add(it, "X", i[0].item())
            self.table.add(it, "Y", i[1].item())
            self.table.add(it, "Alpha", alpha)
            self.table.add(it, "Z min", min(img))
            self.table.add(it, "Z max", max(img))
            self.table.add(it, "Z end", img[-1])
            self.table.add(it, "X min", pts[idx][0])
            self.table.add(it, "Y min", pts[idx][1])
            if [min(img), idx] < best_dist:
                best_dist = [min(img), idx]
                best_init = i
            it+=1
        self.table.show()
        self.show_plots(alpha, best_init)


# RMS Prop

In [ ]:
"""! @brief Declaration of the class RMSProp"""

class RMSProp:
    """! RMS prop algorithm class
    The Gradient Descent algorithm, the calibration of its parameters with Optuna and analysis of the best run.
    """

    def __init__(self, fun: Function, table : Table):
        """! Initialize the RMSProp class

        @param fun The function to be optimized.
        @param table A panda table to show data of the runs of the algorithm.

        @return An instance of the RMS prop class with the function and the table
        """
        #Number of iterations
        self.T = 25

        # Function to be optimized
        self.fun = fun

        # Initial point of the algorithm
        self.x0 = torch.tensor([2, 1])

        # Table to store results
        self.table = table

    def descent(self, x_t, rho: float, gamma: float):
        pts = []
        img = []
        eps = 1e-15 # maybe smaller?
        si = torch.zeros(2, dtype=torch.float32)
        for i in range(self.T):
            grad = self.fun.gradient(x_t)
            si = si * gamma + grad * grad * (1 - gamma)
            alpha = rho / torch.sqrt(si + eps)
            x_t = x_t - alpha * grad
            pts.append(x_t.tolist())
            val=self.fun.f(x_t[0], x_t[1], torch)
            img.append(val.item())
        return pts, img

    def objetive(self, trial):
        """! Function for Optuna to optimize.
        @param trial An Optuna class.

        @return value of the function in this trial.
        """

        # Rho suggested by Optuna in the range [1e-11, 1]
        rho = trial.suggest_float("rho", 1e-11, 1, log=True)

        # Gamma suggested by Optuna in the range [1e-11, 1]
        gamma = trial.suggest_float("gamma", 1e-11, 1, log=True)

        # Visited values of the function.
        _, z= self.descent(self.x0, rho, gamma)

        # Return the last value
        return z[-1]

    def study(self):
        """ ! function to calibrate parameters with Optuna.
        @return The best parameters found as a dictionary.
        """

        # Arbitrary initial point for the algorithm.
        self.x0=torch.tensor([5.0, 5.0])

        # To minimize the function.
        stud = optuna.create_study(direction="minimize")

        # Objective function and number of trials.
        stud.optimize(self.objetive, n_trials=50)

        # Print the best parameters found.
        print(stud.best_params)

        # Visualizations of the process provided by Optuna.
        optuna.visualization.plot_optimization_history(stud).show()
        optuna.visualization.plot_slice(stud, params=["rho", "gamma"]).show()

        # Return the best parameters found
        return stud.best_params

    def show_plots(self, gamma, rho, init):
        """! Show different plots for the best configuration found
        @param alpha The hyperparameter gamma of the algorithm.
        @param rho The hyperparameter rho of the algorithm.
        @param init The initial point of the algorithm.
        """
        # Executes the optimization algorithm
        pts, img = self.descent(init, rho, gamma)

        # Plots contour with optimization trajectory
        plot.make_contour_p(
            self.fun,
            [x[0] for x in pts],
            [x[1] for x in pts],
            "RMSProp",
            self.fun.n
        )

        # Plots surface with optimization points
        plot.make_surface_p(
            self.fun,
            [x[0] for x in pts],
            [x[1] for x in pts],
            img,
            "RMSProp",
            self.fun.n
        )

        # Plots learning curve (function value vs iterations)
        plot.make_plot(
            range(len(pts)),
            img,
            "RMSProp",
            self.fun.n
        )

        # Displays all generated plots
        plot.show()

    def run(self):
        """! Run 10 times the algorithm with the optimized parameters and show the best one with graphs.
        """
        best = self.study()
        rho, gamma = best["rho"], best["gamma"]
        it = 0
        best_dist = [math.inf, math.inf]
        best_init = torch.tensor([math.inf, math.inf])
        for i in INITIAL_POINTS:
            points, images = self.descent(i, rho, gamma)
            idx=images.index(min(images))
            self.table.add(it, "X", i[0].item())
            self.table.add(it, "Y", i[1].item())
            self.table.add(it, "Rho", rho)
            self.table.add(it, "Gamma", gamma)
            self.table.add(it, "Z min", min(images))
            self.table.add(it, "Iteration", idx)
            self.table.add(it, "Z max", max(images))
            self.table.add(it, "Z end", images[-1])
            self.table.add(it, "X min", points[idx][0])
            self.table.add(it, "Y min", points[idx][1])
            if [min(images), idx] < best_dist:
                best_dist = [min(images), idx]
                best_init = i
            it += 1
        self.table.show()
        self.show_plots(gamma, rho, best_init)


# Análisis con Optuna

In [ ]:
# Initializes Gradient Descent for function f0 and runs the algorithm

# Creates a results table with relevant metrics.
GD0 = GradientDescent(
    func0,
    Table(pd.DataFrame(), ["X", "Y", "Alpha", "Iteration", "Z min", "Z max", "Z end", "X min", "Y min"])
)

# Executes the Gradient Descent algorithm
GD0.run()

In [ ]:
# Initializes Gradient Descent for function f1 and runs the algorithm

# Creates a results table with relevant metrics
# GD1 = GradientDescent(func1, Table(pd.DataFrame(), ["X","Y","Alpha","Iteration","Z min", "Z max","Z end"]))

# Executes the Gradient Descent algorithm
# GD1.run()

In [ ]:
# Initializes Gradient Descent for function f2 and runs the algorithm

# Creates a results table with relevant metrics
# GD2 = GradientDescent(func2, Table(pd.DataFrame(), ["X","Y","Alpha","Iteration","Z min", "Z max","Z end"]))

# Executes the Gradient Descent algorithm
# GD2.run()

In [ ]:
# Initializes RMSProp for function f0 and runs the algorithm

# Creates a results table with relevant metrics.
# RMS0 = RMSProp(func0, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end", "X min", "Y min"]))

# Executes the RMSProp algorithm
#RMS0.run()

In [ ]:
# Initializes RMSProp for function f1 and runs the algorithm

# Creates a results table with relevant metrics.
# RMS1 = RMSProp(func1, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end"]))

# Executes the RMSProp algorithm
# RMS1.run()

In [ ]:
# Initializes RMSProp for function f2 and runs the algorithm

# Creates a results table with relevant metrics.
# RMS2 = RMSProp(func2, Table(pd.DataFrame(), ["X","Y","Rho","Gamma","Iteration","Z min", "Z max","Z end"]))

# Executes the RMSProp algorithm
# RMS2.run()